# Notebook 04a — How the agents work, and what authorises them

**ATLAS: Aligned Three-Layer Architecture for Semantics**
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

You have seen the MCP servers (02), the registry that lists the agents (03), and the
GraphQL API the UI talks to (04). This notebook answers the question a banker actually
asks the first time they use the application:

> When I type a question into the app, what happens between my keystroke and the answer —
> and what stops the language model from simply inventing a reply?

That question has two halves, and this notebook teaches both. The first half is *what the
agents do*: the three agents behind the Wholesale and Wealth screens, and the precise,
bounded thing each one is allowed to do. The second half is *what lets them do it*: the
identity and permission model that carries a request from the browser down to Neptune,
and the deliberate limits that keep the whole path auditable.

Everything below describes the system as it runs today. Where the system cannot yet do
something, the notebook says so plainly and points at what would close the gap. A
teaching notebook that overstates its system is worse than no notebook, so the live cells
here call the real agents and show you their real output — including the case where an
agent refuses to answer.


## The concept: three narrow agents, one strict identity

### The three agents and the one thing each is allowed to do

The intelligent layer is not one model that does everything. It is three agents, each
deliberately narrow, because narrowness is what makes the behaviour explainable to a
regulator.

The first agent, nl-to-sparql-agent, is what answers "ask the graph." It does not write
SPARQL freely. It embeds your question with a Bedrock Titan embedding, compares it against
a fixed library of validated query templates, and runs the closest template only if the
match clears a similarity threshold. If nothing clears the bar it returns a no-match
result rather than guessing. Free SPARQL generation is forbidden by construction. The set
of templates is therefore the contract: it is exactly the set of questions the system can
answer, and you will read that set live from the same file the agent uses, a few cells
below.

The second agent, referral-rationale-drafter, writes the narrative a banker reviews before
a referral is routed. It first queries the semantic layer for the household and for the
specific signals in play, then asks a Bedrock Claude model to draft a rationale grounded
in those facts. Its output is flagged probabilistic and requires-human-review, every time.
The banker edits and approves; only then does the deterministic routing step run. The
model drafts; it never decides.

The third agent, conversational-context-manager, backs the Wealth UI conversation. It
wraps the first agent, so it inherits the same template-bounded behaviour. It is built to
carry conversation memory across turns, but that memory is not wired today, so each
question is answered on its own. The notebook will show you that fact directly, read from
the response, rather than asking you to take it on faith.

### What authorises them: service identity, not the user's token

A request starts as a person. The browser holds a Cognito sign-in token, and AppSync
checks that token at the edge before any resolver runs. But the agents do not run as the
person. Once a request crosses from AppSync into the agent layer, it runs under an AWS
identity, and each hop is signed with that identity and authorised by IAM.

This is a design choice, and it is worth understanding why. The alternative — carrying the
user's token all the way down so each agent acts "as the user" — is not available here:
there is no runtime-to-runtime call that forwards a user token between AgentCore runtimes,
and there is no token-relay plumbing to build one on. So the system uses service identity:
the runtimes are configured with an IAM authorizer, and every caller signs its request
with SigV4 and must hold an explicit IAM grant to invoke the next hop.

There are four such grants, and together they are the whole chain. The GraphQL resolver is
granted permission to invoke the three agents directly by ARN. The conversational agent is
granted permission to invoke the natural-language agent it wraps. Each agent that reaches
the graph is granted permission to invoke the SPARQL MCP server. And the agents that call a
language model are granted permission to invoke exactly the Bedrock model they use, and no
other. Nothing in the chain can reach a resource it was not explicitly handed.

### What enforces the rules: three layers that are real, and one that is not yet

It is common to describe this kind of system as having four enforcement layers. Be precise
about which ones are actually running. IAM authorises every service-to-service hop, as
just described. Cognito authenticates the person at the AppSync edge. SHACL validates the
shape of a decision before it is allowed to persist — the routing step refuses a
non-conformant decision rather than writing it. Those three are live and you can trace each
in the code. A fourth layer, Lake Formation row and column scoping, is described in the
architecture but is deferred: it is not deployed, and the read path is scoped at the edge
by persona class rather than filtered per row downstream. Teaching the system honestly
means naming the three that run and marking the fourth as future work.


## Show me: the agents, live

The cells below call the real agents through the GraphQL API and show their real output.
They need two things from your environment: the AppSync endpoint and a Cognito access
token for a signed-in user. This is the same precondition the build and acceptance
notebooks have — if you have not deployed the stack and obtained a token, the cells will
say so and you can read the expected output in the markdown instead. Nothing here is
simulated; an undeployed run simply has nothing real to call.


In [ ]:
import json, os, urllib.request, re

# Two inputs from your deployed stack (the same pattern as 05a / 07):
#   APPSYNC_ENDPOINT  — the AtlasWorkshop2 AppSyncEndpoint output
#   ATLAS_BEARER_TOKEN — a Cognito ACCESS token for a signed-in user
APPSYNC_ENDPOINT = os.environ.get("APPSYNC_ENDPOINT", "")
BEARER = os.environ.get("ATLAS_BEARER_TOKEN", "")
LIVE = bool(APPSYNC_ENDPOINT and BEARER)

def gql(query, variables=None):
    """POST a GraphQL operation to AppSync with the user's Cognito token."""
    body = json.dumps({"query": query, "variables": variables or {}}).encode()
    req = urllib.request.Request(APPSYNC_ENDPOINT, data=body, method="POST")
    req.add_header("Content-Type", "application/json")
    req.add_header("Authorization", BEARER)
    with urllib.request.urlopen(req, timeout=90) as r:
        return json.loads(r.read())

print("LIVE — will call the real API." if LIVE
      else "NOT LIVE — set APPSYNC_ENDPOINT + ATLAS_BEARER_TOKEN to run the cells against your stack.\n"
           "The markdown below each cell describes the real output you would see.")


### The contract: the questions the system can actually answer

The natural-language agent answers only what its template library covers. The cell reads
that library directly from `ground-truth.yaml` — the same file the agent matches against
and the same file the `suggestedQuestions` API field serves to the UI — so what you see
here cannot drift from what the agent will actually accept.


In [ ]:
# The 10 templates ARE the contract. Read them live from the agent's own source file
# (WS1-owned, read-only here) so this list can never drift from what the agent answers.
GROUND_TRUTH = "../../../agentic-semantic-layer/prompts/ground-truth.yaml"
with open(GROUND_TRUTH) as f:
    body = f.read()
questions = re.findall(r'^\s*-\s*question:\s*"(.+?)"\s*$', body, flags=re.MULTILINE)
print(f"{len(questions)} questions the system can answer:\n")
for i, q in enumerate(questions, 1):
    print(f"{i:2}. {q}")


### A question it can answer: real rows, and the SPARQL that produced them

`askGraph` routes to nl-to-sparql-agent. A matched question comes back with the rows from
Neptune, the validated SPARQL that ran, and the id of the template that matched. The
answer is not the model's prose — it is the graph's data, reached through a query you can
inspect.


In [ ]:
ASK = """query($q: String!) { askGraph(question: $q) { status sparql templateId result } }"""

if LIVE:
    r = gql(ASK, {"q": "Which customers have no wealth advisor assigned?"})["data"]["askGraph"]
    rows = r["result"] if isinstance(r["result"], list) else json.loads(r["result"] or "[]")
    print("status   :", r["status"])
    print("template :", r["templateId"])
    print("rows     :", len(rows))
    print("first row:", rows[0] if rows else "(none)")
    print("\nthe SPARQL that ran:\n", (r["sparql"] or "")[:400])
else:
    print('Expected (live): status "success", a real templateId, a non-empty row list,')
    print('and the validated SPARQL the template produced — the answer is graph data,')
    print('not generated text.')


### A question it cannot answer: it refuses, it does not invent

This is the property that matters most for a regulated setting. Ask something outside the
template library and the agent returns a no-match result with no rows. It does not
improvise a query or fabricate an answer. The honest empty result is the feature.


In [ ]:
if LIVE:
    r = gql(ASK, {"q": "what is the weather on mars today"})["data"]["askGraph"]
    print("status:", r["status"], "| rows:", r["result"])
    print("\n-> The agent declined rather than guessing. The UI responds by showing the")
    print("   answerable questions above, never a made-up answer.")
else:
    print('Expected (live): status "no_template_match", result []. No fabricated answer.')


### The conversation: real answers, and the single-turn truth in the data

`converse` routes to conversational-context-manager. It returns the same kind of real
result as `askGraph`, plus a `priorTurns` count. That count is always zero today: the
agent's memory is not wired, so each question stands alone. You are reading the limit from
the response, not from a promise in the copy.


In [ ]:
CONVERSE = """mutation($q: String!, $s: String!) {
  converse(question: $q, sessionId: $s) { status priorTurns sparql result }
}"""

if LIVE:
    r = gql(CONVERSE, {"q": "Which customers have no wealth advisor assigned?",
                       "s": "demo-session-1"})["data"]["converse"]
    rows = r["result"] if isinstance(r["result"], list) else json.loads(r["result"] or "[]")
    print("status    :", r["status"])
    print("priorTurns:", r["priorTurns"], "  <- always 0: the conversation is single-turn today")
    print("rows      :", len(rows))
else:
    print('Expected (live): status "success", real rows, priorTurns 0 — single-turn,')
    print('surfaced in the data rather than claimed in the interface.')


## Verification: what you just saw, and what it proves

If you ran the cells live, you saw four things, and each one is a claim this notebook
makes turned into evidence.

The template list came from the agent's own source file, so the questions shown are
exactly the questions the agent accepts — the contract is not a description, it is the
data. The matched question returned real rows and the SPARQL that produced them, so the
answer is the graph's, reached through an inspectable query, not the model's invention.
The unmatched question returned a no-match with no rows, so the system's response to "I
cannot answer this" is to say so. And the conversation returned `priorTurns` of zero, so
the single-turn limitation is something you can read off the response rather than infer.

If you read along instead of running live, the expected output under each cell describes
the same four facts. Either way, none of it is simulated — an undeployed run has nothing
real to call, and the notebook says that rather than printing a fake result.


## What this teaches, including where the system stops

The deepest lesson here is that the limits are not embarrassments to hide; they are the
properties that make the system trustworthy, and each one has a reason and a path forward.

The conversation is single-turn because the agent's memory calls do not map to real
AgentCore Memory operations, so they quietly do nothing. The honest fix is to rewrite that
agent against the real event APIs (CreateEvent and RetrieveMemoryRecords); until then,
`priorTurns` reads zero and the interface says single-turn. The answers are scoped by
persona class and not by individual advisor, which is why the conversation speaks of "the
graph" and the book of business rather than "your clients." Lifting that needs the
advisor's own identity carried into the query, templates that filter on it, and the
coverage data to join against — none of which is in place yet. And the natural-language
path is bounded by ten templates rather than open generation. That is the regulatory
posture, not a shortcoming: a fixed, validated set of queries is deterministic and
auditable in a way that free generation is not.

A practical note for anyone who edits an agent. Changing an agent's code is not the same as
changing the stack around it. The agent code ships as a packaged artifact, so a code change
means rebuilding that artifact and updating the runtime to pick it up — a plain `cdk
deploy` will not notice it, because the artifact's storage key has not changed. A change to
IAM, to the GraphQL schema, or to the UI is the ordinary `cdk deploy` path. The deploy
runbook in notebook 08 covers the full sequence.

A few capabilities are deliberately not finished, and they are named here rather than
implied complete. Multi-turn memory is future work, as above. Per-user scoping is future
work. The detect-signals mutation still routes through an older invocation path that does
not reach its agent and needs the same direct-ARN treatment the three agents here received.
The compliance banner shows the correct non-tipping-off wording but is driven by an
illustrative flag rather than a real per-entity compliance state. None of these is wired to
a fabrication in the interface; each is marked for what it is.

This notebook closes the slice about how the agents and their authorisation work. It does
not close the larger application-readiness work — the hosted-UI login flow, the CloudFront
routing for the single-page apps, and the resolver and concept-loader refinements remain.
What you can now do that you could not before is trace a single question from a keystroke,
through the edge check and the four signed hops, to a validated query against the graph,
and explain at each step both what runs and why it is allowed to.
